In [1]:
import cv2 # Import OpenCV for video processing
import numpy as np # Import NumPy for numerical operations
from ultralytics import YOLO # Import YOLO model from ultralytics
from collections import defaultdict # Import defaultdict for easier dictionary handling
from pathlib import Path # Import Path for file path management
import math

In [2]:

PROJECT_ROOT = Path.cwd().parent.resolve() # Répertoire racine du projet
VIDEO_OUTPUT_DIR = PROJECT_ROOT / "data" # Répertoire des vidéos de sortie
VIDEO_PATH = VIDEO_OUTPUT_DIR / "test_lyon.mp4" # Chemin de la vidéo d'entrée
MODEL_PATH = (PROJECT_ROOT / "results" / "yolov8n_vehicle_detection2" / "weights" / "best.pt").resolve() # Chemin du modèle entraîné
DEVICE = "cpu"  # ou "cuda"
# ======================
# PARAMÈTRES
# ======================
CONF = 0.25 # seuil de confiance
SMOOTH_N = 10   # lissage vitesse (sliding) window size
LOWER_ZONE_RATIO = 0.3  # zone basse de l'image pour la calibration

## Calibration of the speed computation (PX TO METERS)

Uses an average length for a selection of N_CARS vehicles (KNOWN_DISTANCE_M=4.5m for an average car). 

In [ ]:
## PLEASE RIGHT-CLICK ON THE VIDEO TO PAUSE/RESUME ##
## AND LEFT-CLICK TO SELECT 2 POINTS FOR CALIBRATION ON EACH CAR ##

# ============================================================
# CALIBRATION PARAMETERS
# ============================================================

KNOWN_DISTANCE_M = 4.5   # average real-world car length (meters)
N_CARS = 10              # number of cars used for calibration

# total number of clicks required (2 points per car)
TOTAL_POINTS = 2 * N_CARS


# ============================================================
# GLOBAL STATE VARIABLES
# ============================================================

points = []   # list of clicked points [(x, y), ...]
mpp = []      # list of meters-per-pixel values (one per car)
paused = False  # video pause state


# ============================================================
# MOUSE CALLBACK FUNCTION
# ============================================================

def on_mouse(event, x, y, flags, param):
    """
    Mouse interactions:
    - RIGHT CLICK  : pause / resume video
    - LEFT CLICK   : select a point for calibration
    """

    global paused

    # --------------------------------------------------------
    # RIGHT CLICK → toggle pause / resume
    # --------------------------------------------------------
    if event == cv2.EVENT_RBUTTONDOWN:
        paused = not paused

    # --------------------------------------------------------
    # LEFT CLICK → select calibration point
    # --------------------------------------------------------
    if event == cv2.EVENT_LBUTTONDOWN:

        # stop accepting clicks once enough points are collected
        if len(points) >= TOTAL_POINTS:
            return

        # store the clicked point
        points.append((x, y))

        # ----------------------------------------------------
        # every pair of points corresponds to one car
        # ----------------------------------------------------
        if len(points) % 2 == 0:

            # retrieve the last two points
            (x1, y1), (x2, y2) = points[-2], points[-1]

            # pixel distance between the two points
            dpx = math.hypot(x2 - x1, y2 - y1)

            # avoid division by zero
            if dpx > 0:
                # compute meters-per-pixel for this car
                mpp.append(KNOWN_DISTANCE_M / dpx)

                # print intermediate result
                print(f"m/px = {mpp[-1]:.6f}")


# ============================================================
# VIDEO INITIALIZATION
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

# safety check
if not cap.isOpened():
    raise RuntimeError("cannot open the video")

# create display window
cv2.namedWindow("calib")

# attach mouse callback to the window
cv2.setMouseCallback("calib", on_mouse)

frame = None  # current video frame


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    # --------------------------------------------------------
    # read a new frame only if video is not paused
    # --------------------------------------------------------
    if not paused or frame is None:
        ret, frame = cap.read()
        if not ret:
            break

    # --------------------------------------------------------
    # image geometry
    # --------------------------------------------------------
    H, W = frame.shape[:2]

    # lower zone threshold (used for consistency with speed computation)
    Y_MIN = int(H * (1 - LOWER_ZONE_RATIO))

    # copy frame for visualization (do NOT draw on original)
    vis = frame.copy()

    # --------------------------------------------------------
    # display number of selected points
    # --------------------------------------------------------
    cv2.putText(
        vis,
        f"{len(points)}/{TOTAL_POINTS}",  # current / total
        (20, 110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # --------------------------------------------------------
    # draw lower zone reference line
    # --------------------------------------------------------
    cv2.line(
        vis,
        (0, Y_MIN),
        (W, Y_MIN),
        (255, 0, 0),
        2
    )

    # --------------------------------------------------------
    # draw already measured segments (one per car)
    # --------------------------------------------------------
    for i in range(0, len(points), 2):
        if i + 1 < len(points):
            cv2.line(
                vis,
                points[i],
                points[i + 1],
                (0, 255, 0),
                2
            )

    # --------------------------------------------------------
    # pause indicator
    # --------------------------------------------------------
    if paused:
        cv2.putText(
            vis,
            "paused",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    # --------------------------------------------------------
    # display frame
    # --------------------------------------------------------
    cv2.imshow("calib", vis)

    # --------------------------------------------------------
    # exit conditions:
    # - ESC key
    # - enough cars measured
    # --------------------------------------------------------
    key = cv2.waitKey(20) & 0xFF
    if key == 27 or len(mpp) == N_CARS:
        break


# ============================================================
# CLEANUP
# ============================================================

cap.release()
cv2.destroyAllWindows()


# ============================================================
# FINAL CALIBRATION RESULT
# ============================================================

# compute final meters-per-pixel as the average over all cars
METER_PER_PIXEL = np.mean(mpp) if len(mpp) > 0 else None

if len(mpp) > 0:
    print("METERS_PER_PIXEL =", METER_PER_PIXEL)
else:
    print("No valid calibration measurement")


m/px = 0.022472
m/px = 0.021626
m/px = 0.033541
m/px = 0.019654
m/px = 0.026334
m/px = 0.029770
m/px = 0.020214
m/px = 0.025133
m/px = 0.023328
m/px = 0.022887
METERS_PER_PIXEL = 0.02449574706793445


## Speed computation (Live)

In [4]:
# ======================
# INIT
# ======================
model = YOLO(MODEL_PATH)    
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

history = defaultdict(list)  # {id: [(frame, cx, cy)]}
frame_id = 0

# ======================
# VITESSE px/s
# ======================
def speed_px_s(track):  # track = [(frame, cx, cy), ...]
    if len(track) < SMOOTH_N + 1:
        return None

    f1, x1, y1 = track[-(SMOOTH_N + 1)]
    f2, x2, y2 = track[-1]

    dt = (f2 - f1) / fps
    if dt <= 0:
        return None

    return np.hypot(x2 - x1, y2 - y1) / dt

# ======================
# LOOP PRINCIPALE
# ======================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    H, W = frame.shape[:2]
    Y_MIN = H * (1 - LOWER_ZONE_RATIO)

    # --- INFERENCE + TRACKING ---
    results = model.track(
        frame,
        persist=True,
        conf=CONF,
        verbose=False,
        device=DEVICE
    )[0]

    # --- DESSIN YOLO STANDARD ---
    annotated_frame = results.plot()

    # --- LIGNE ZONE BASSE ---
    cv2.line(
        annotated_frame,
        (0, int(Y_MIN)),
        (W, int(Y_MIN)),
        (255, 0, 0),
        2
    )

    # --- CENTRE + VITESSE ---
    if results.boxes.id is not None:
        ids = results.boxes.id.cpu().numpy().astype(int)
        xyxy = results.boxes.xyxy.cpu().numpy()

        for obj_id, (x1, y1, x2, y2) in zip(ids, xyxy):
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2

            # vitesse calculée uniquement en zone basse
            if cy < Y_MIN:
                continue

            history[obj_id].append((frame_id, cx, cy))
            v = speed_px_s(history[obj_id])

            # centre
            cv2.circle(annotated_frame, (int(cx), int(cy)), 4, (0, 0, 255), -1)

            # vitesse
            if v is not None:
                v_kmh = v * METER_PER_PIXEL * 3.6
                cv2.putText(
                    annotated_frame,
                    f"{v_kmh:.1f} km/h",
                    (int(x1), int(y2) + 18),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 0, 255),
                    2
    )


    # --- AFFICHAGE ---
    cv2.imshow("YOLO speed px/s", annotated_frame)
    if cv2.waitKey(1) == 27:  # ESC
        break

cap.release()
cv2.destroyAllWindows()

## Speed computation (Video output) - no display

In [ ]:
# ======================
# INIT
# ======================
model = YOLO(MODEL_PATH)    
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(
    "video_annotated.mp4",
    fourcc,
    fps,
    (width, height)
)

history = defaultdict(list)  # {id: [(frame, cx, cy)]}
frame_id = 0

# ======================
# VITESSE px/s
# ======================
def speed_px_s(track):  # track = [(frame, cx, cy), ...]
    if len(track) < SMOOTH_N + 1:
        return None

    f1, x1, y1 = track[-(SMOOTH_N + 1)]
    f2, x2, y2 = track[-1]

    dt = (f2 - f1) / fps
    if dt <= 0:
        return None

    return np.hypot(x2 - x1, y2 - y1) / dt

# ======================
# LOOP PRINCIPALE
# ======================
while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame_id += 1

    H, W = frame.shape[:2]
    Y_MIN = H * (1 - LOWER_ZONE_RATIO)

    # --- INFERENCE + TRACKING ---
    results = model.track(
        frame,
        persist=True,
        conf=CONF,
        verbose=False,
        device=DEVICE
    )[0]

    # --- DESSIN YOLO STANDARD ---
    annotated_frame = results.plot()

    # --- LIGNE ZONE BASSE ---
    cv2.line(
        annotated_frame,
        (0, int(Y_MIN)),
        (W, int(Y_MIN)),
        (255, 0, 0),
        2
    )

    # --- CENTRE + VITESSE ---
    if results.boxes.id is not None:
        ids = results.boxes.id.cpu().numpy().astype(int)
        xyxy = results.boxes.xyxy.cpu().numpy()

        for obj_id, (x1, y1, x2, y2) in zip(ids, xyxy):
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2

            # vitesse calculée uniquement en zone basse
            if cy < Y_MIN:
                continue

            history[obj_id].append((frame_id, cx, cy))
            v = speed_px_s(history[obj_id])

            # centre
            cv2.circle(annotated_frame, (int(cx), int(cy)), 4, (0, 0, 255), -1)

            # vitesse
            if v is not None:
                v_kmh = v * METER_PER_PIXEL * 3.6
                cv2.putText(
                    annotated_frame,
                    f"{v_kmh:.1f} km/h",
                    (int(x1), int(y2) + 18),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 0, 255),
                    2
    )


    # --- export ---
    out.write(annotated_frame)


cap.release()
out.release()
